# 42. Topology-mixture calibration training from checkpoint 002.23

**Goal:** preserve notebook 39's signed-prior calibration-learning objective, model architecture, three-view loss, train/validation/test datasets, checkpointing, and inspection workflow, while replacing the **Z-yaw curriculum** with a mixture of physical sensor topologies.

- **50%** KAIST's actual **one IMU + two LiDAR** streams.
- **25%** source IMU + synthetic second IMU + **right** real LiDAR (left LiDAR removed).
- **25%** source IMU + synthetic second IMU + **left** real LiDAR (right LiDAR removed).

The synthetic IMU is a **reoriented version of KAIST's real gyro/accelerometer**, not an interpretation of the removed LiDAR. The copied IMU has no independent physical information at zero synthetic noise. The native LiDAR modality is never counterfeited.

**Important interface:** This notebook expects `SensorTopologyAugmentationConfig`, `SensorTopologyAugmenter`, and the topology stage in `AugmentationPipeline` from the Python augmentation implementation. It validates those preconditions early. For the requested **per-window** proportions, it samples each example independently and groups examples into homogeneous sub-batches, since `WindowBatch` cannot represent differing stream schemas within one batch.

Keep the calibration-prior perturbation **exactly one sensor per sample**, as in 002.23. Initial model weights: **002.23 `best.pt`**, fresh optimizer. Common-frame **Z yaw at fixed probability** for maintenance only; **no yaw curriculum**. True-calibration rotations remain independent for each of the three final active sensors.

## 1. Imports and repository root

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the nn-obs-calib repository root containing src/obscalib.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import SensorTopologyAugmentationConfig, AugmentationConfig, CalibrationEventConfig, CommonBodyFrameRandomizationConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations import PerturbationDistribution
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber, apply_prior_perturbation
from obscalib.calibration.state import CalibrationState
from obscalib.calibration.update import CalibrationUpdater
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch
from obscalib.geometry.lie import se3_log
from obscalib.geometry.processing import GeometryProcessor
from obscalib.models.pooled_calibration_model import PooledCalibrationModel
from obscalib.pipeline.window_step import WindowStep
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint
from obscalib.training.metrics_logger import MetricsLogger
import gc
from dataclasses import fields, is_dataclass, replace

print("Repository root:", ROOT)
print("PyTorch:", torch.__version__)

## 2. Topology-mixture training configuration and checkpoint 002.23

In [ ]:
RUN_NAME = "kaist_pooled_calibration_paired_002.27_mixed_imu_lidar_topology"

# Specify the exact previous successful checkpoint; this is a weights-only initialization.
# This exact successful 002.23 checkpoint initializes only the model weights.
SOURCE_RUN_NAME = "kaist_pooled_calibration_paired_002.23_yaw_only_crossframe_curriculum"
RESUME_CHECKPOINT = ROOT / "runs" / SOURCE_RUN_NAME / "best.pt"

RANDOM_SEED = 12345
TRAIN_PAIR_SEED = 23456
VALIDATION_PAIR_SEED = 34567
TEST_PAIR_SEED = 45678
VISUALIZATION_PAIR_SEED = 56789

DEVICE_NAME = "cuda"
DEVICE = torch.device("cuda" if DEVICE_NAME == "auto" and torch.cuda.is_available() else DEVICE_NAME)
MODEL_DTYPE = torch.float32

CALIBRATION_KEYS = ("imu", "lidar", "lidar_right")
CALIBRATION_CONTEXT_DIM = 7

WINDOW_DURATION_S = 5.0
WINDOW_STRIDE_S = 5.0
MAX_SAMPLES_PER_SENSOR = 700

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1  # Effective optimizer batch 8; topology groups retain weighted per-example contributions
NUM_WORKERS = 0
DROP_LAST_TRAIN = True

NUM_EPOCHS = 120
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
CHECKPOINT_EVERY_EPOCHS = 3
PRINT_EVERY_BATCHES = 500
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE_EPOCHS = 8

MEASUREMENT_DIM = 6
TOKEN_INPUT_DIM = MEASUREMENT_DIM + 2
D_MODEL = 128
N_HEADS = 8
NUM_TRANSFORMER_LAYERS = 12
DIM_FEEDFORWARD = 512
TRANSFORMER_DROPOUT = 0.0
NUM_SUMMARY_TOKENS = 1

POOLER_N_HEADS = 8
POOLER_DROPOUT = 0.0
MLP_HIDDEN_DIMS = (256, 128)
SHARED_FEATURE_DIM = 128
MLP_DROPOUT = 0.0
HEAD_HIDDEN_DIMS = (128, 64)
HEAD_DROPOUT = 0.0

CORRECTION_LOSS_TYPE = "mse"
ROTATION_LOSS_SCALE_RAD = np.deg2rad(5.0)
TRANSLATION_LOSS_SCALE_M = 0.05
TIME_OFFSET_LOSS_SCALE_S = 0.050
ROTATION_LOSS_WEIGHT = 2.0
TRANSLATION_LOSS_WEIGHT = 1.0
TIME_OFFSET_LOSS_WEIGHT = 1.0
EQUIVARIANCE_LOSS_WEIGHT = 0.05  # Do NOT increase indiscriminately: zero outputs are already equivariant

# Extra gradient on the physically perturbed sensor. This is more direct than pair loss at countering near-zero corrections.
ACTIVE_ROTATION_SUPERVISION_WEIGHT = 0.75
ROTATED_ACTIVE_MULTIPLIER = 2.0  # Prioritize genuinely perturbed sensors in the rotated view

# Three instead of two forward passes when enabled; one extra *supervised* opposite-prior view.
# Set USE_SIGNED_PAIR=False for a two-view control, without changing the rest of the experiment.
USE_SIGNED_PAIR = True
OPPOSITE_VIEW_SUPERVISION_WEIGHT = 0.5  # Relative view weight; positive and rotated each have weight 1.
PAIR_LOSS_WEIGHT = 0.10 if USE_SIGNED_PAIR else 0.0  # Cross-frame signed ROTATION gap, not the old original-frame pair


# Fresh optimizer/scheduler from the successful 002.23 yaw-robust checkpoint; this is NOT an exact resume.
COSINE_MIN_LEARNING_RATE = 1e-6
RUN_FULL_TRAINING = True

CHANGE_EVENT_LOSS_WEIGHT = 0.0
CHANGE_TIME_LOSS_WEIGHT = 0.0

# One paired-prior distribution for every example; no fine/coarse branch.
PAIR_ROTATION_MAX_RAD = np.deg2rad(60.0)
PAIR_ROTATION_SCALE_RAD = np.deg2rad(30.0)
PAIR_TIME_OFFSET_MAX_S = 0.500
PAIR_TIME_OFFSET_SCALE_S = 0.250
STUDENT_T_DEGREES_OF_FREEDOM = 3
PAIR_TRANSLATION_MAX_M = 0.02  # Legacy disabled generic-prior config only.

# Physically consistent true-calibration augmentation, independent of the prior error.
TRUE_CALIBRATION_ROTATION_MAX_RAD = np.deg2rad(180.0)
TRUE_CALIBRATION_TIME_OFFSET_MAX_S = 0.030

# NEW: per-physical-window topology mixture. One per-window categorical draw, then
# grouping by topology before running the original AugmentationPipeline.
TOPOLOGY_ORIGINAL_PROBABILITY = 0.50
TOPOLOGY_REPLACE_LEFT_PROBABILITY = 0.25
TOPOLOGY_REPLACE_RIGHT_PROBABILITY = 0.25
TOPOLOGY_SYNTHETIC_MOUNT_MAX_DEG = 180.0
TOPOLOGY_SYNTH_GYRO_NOISE_STD = 0.0    # rad/s, zero isolates modality topology
TOPOLOGY_SYNTH_ACCEL_NOISE_STD = 0.0   # m/s^2
TOPOLOGY_MODES = ("original", "replace_lidar", "replace_lidar_right")
TOPOLOGY_PROBABILITIES = (TOPOLOGY_ORIGINAL_PROBABILITY, TOPOLOGY_REPLACE_LEFT_PROBABILITY, TOPOLOGY_REPLACE_RIGHT_PROBABILITY)
assert all(probability >= 0.0 for probability in TOPOLOGY_PROBABILITIES)
assert abs(sum(TOPOLOGY_PROBABILITIES) - 1.0) < 1e-10

# Preserve some previously learned yaw robustness without scheduling difficulty.
COMMON_BODY_FRAME_PROBABILITY = 0.25
COMMON_BODY_FRAME_ROTATION_AXES = (False, False, True)
COMMON_BODY_FRAME_MAX_ROTATION_DEG = 180.0
VALIDATE_WITH_FULL_SELECTED_AXIS_RANGE = True

# Held-out evaluation uses each topology separately with fixed, reproducible seeds.
TOPOLOGY_EVAL_MODES = TOPOLOGY_MODES

if not RESUME_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Source checkpoint not found: {RESUME_CHECKPOINT}. Set SOURCE_RUN_NAME to the saved successful run.")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Run:", RUN_NAME)
print("Topology-mixture one-window training (BPTT disabled); fixed yaw probability:", COMMON_BODY_FRAME_PROBABILITY)
print("Initializing from:", RESUME_CHECKPOINT)
print("Calibration order:", CALIBRATION_KEYS)
print(f"Single prior: rotation maximum={np.rad2deg(PAIR_ROTATION_MAX_RAD):g}°, Student-t scale={np.rad2deg(PAIR_ROTATION_SCALE_RAD):g}°")
print(f"Single prior: tau maximum={PAIR_TIME_OFFSET_MAX_S * 1000:g} ms, Student-t scale={PAIR_TIME_OFFSET_SCALE_S * 1000:g} ms")
print(f"True calibration: rotation maximum={np.rad2deg(TRUE_CALIBRATION_ROTATION_MAX_RAD):g}°, tau maximum={TRUE_CALIBRATION_TIME_OFFSET_MAX_S * 1000:g} ms")

print("Equivariance weight:", EQUIVARIANCE_LOSS_WEIGHT, "rotated active multiplier:", ROTATED_ACTIVE_MULTIPLIER)
print("Per-window topology mixture:", dict(zip(TOPOLOGY_MODES, TOPOLOGY_PROBABILITIES)))
print("Static common-body Z-yaw max [deg]:", COMMON_BODY_FRAME_MAX_ROTATION_DEG, "probability:", COMMON_BODY_FRAME_PROBABILITY)

print("Signed opposite-prior branch:", USE_SIGNED_PAIR, "pair loss weight:", PAIR_LOSS_WEIGHT, "active emphasis:", ACTIVE_ROTATION_SUPERVISION_WEIGHT)

## 3. KAIST splits and DataLoaders (unchanged from notebook 38)

In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

# Right LiDAR -> body, in the same convention as T_B_L for the left LiDAR.
T_B_L_right = np.array([
    [-0.512152,  0.699241, -0.498761, -0.449885],
    [-0.494811, -0.714859, -0.494104, -0.416713],
    [-0.702041, -0.0062642, 0.712109,  1.91294],
    [ 0.0,       0.0,       0.0,       1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0, T_B_L_right=T_B_L_right, tau_lidar_right_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)


In [ ]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT, lidar_right_pose_filename="lidar_map_poses_vlp_right.csv")

# The 3-sensor experiment must never silently fall back to 2 sensors.
for spec in all_specs:
    if spec.lidar_right_pose_csv is None or not spec.lidar_right_pose_csv.is_file():
        raise FileNotFoundError(f"Missing required right-LiDAR pose CSV for {spec.name}: {spec.lidar_right_pose_csv}")
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

for split_name, dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    if tuple(dataset.calibration_keys) != CALIBRATION_KEYS:
        raise AssertionError(f"{split_name} dataset has calibration keys {dataset.calibration_keys}, expected {CALIBRATION_KEYS}")

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())


In [ ]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))


## 4. Physical augmentation with mixed sensor topology and one erroneous prior

One topology is selected per **physical window** before later augmentation. The real IMU is copied and re-expressed as a second, physically co-located IMU when a LiDAR is replaced. Independent true-calibration and clock augmentation act on all active sensors afterward. The generic pipeline prior is disabled; the separate paired prior still corrupts **only one** sensor.

In [ ]:
# Augment one physical installation, then sample one prior; apply common-frame G later to a complete view.
# The generic prior perturbation stays disabled to avoid modifying the shared truth.
context_augmentation_config = AugmentationConfig(
    sensor_topology=SensorTopologyAugmentationConfig(enabled=False),
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    common_body_frame=CommonBodyFrameRandomizationConfig(enabled=False, probability=0.0),
    true_calibration=TrueCalibrationAugmentationConfig(
        enabled=True,
        probability=1.0,
        rotation_probability=1.0,
        translation_probability=0.0,
        time_offset_probability=1.0,
        rotation=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_ROTATION_MAX_RAD),
        time_offset=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_TIME_OFFSET_MAX_S),
    ),
    prior_perturbation=PriorPerturbationConfig(enabled=False, probability=1.0, rotation=PerturbationMagnitudeConfig(maximum=PAIR_ROTATION_MAX_RAD), translation=PerturbationMagnitudeConfig(maximum=PAIR_TRANSLATION_MAX_M), time_offset=PerturbationMagnitudeConfig(maximum=PAIR_TIME_OFFSET_MAX_S), one_calibration_per_sample=True, one_rotation_axis_per_sample=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)

pair_prior_config = PriorPerturbationConfig(
    enabled=True,
    probability=1.0,
    rotation=PerturbationMagnitudeConfig(maximum=PAIR_ROTATION_MAX_RAD, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PAIR_ROTATION_SCALE_RAD, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    translation=None,
    time_offset=PerturbationMagnitudeConfig(maximum=PAIR_TIME_OFFSET_MAX_S, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PAIR_TIME_OFFSET_SCALE_S, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    one_calibration_per_sample=True,
    one_rotation_axis_per_sample=False,
    one_perturbation_type_per_sample=False,
)

context_augmentation_pipeline = AugmentationPipeline(context_augmentation_config)
pair_prior_perturber = CalibrationPriorPerturber(pair_prior_config)

print("Shared physical/context augmentation:", context_augmentation_config)
print("Paired prior:", pair_prior_config)
print("Common-frame stage disabled in physical pipeline:", context_augmentation_config.common_body_frame)
from obscalib.augmentations.common_body_frame import CommonBodyFrameRandomizer

# Three preconstructed homogeneous pipelines. Every batch item independently chooses
# a mode, and items of the same mode are batched together before entering one pipeline.
def topology_config_for_mode(mode: str) -> SensorTopologyAugmentationConfig:
    kwargs = dict(
        enabled=(mode != "original"), replacement_probability=1.0,
        replace_first_lidar_probability=1.0 if mode == "replace_lidar" else 0.0,
        maximum_mount_rotation_deg=TOPOLOGY_SYNTHETIC_MOUNT_MAX_DEG,
        synthetic_gyro_noise_std=TOPOLOGY_SYNTH_GYRO_NOISE_STD,
        synthetic_accel_noise_std=TOPOLOGY_SYNTH_ACCEL_NOISE_STD,
    )
    if mode not in TOPOLOGY_MODES:
        raise ValueError(f"Unknown topology mode: {mode!r}")
    return SensorTopologyAugmentationConfig(**kwargs)


context_augmentation_pipelines = {
    mode: AugmentationPipeline(replace(context_augmentation_config, sensor_topology=topology_config_for_mode(mode)))
    for mode in TOPOLOGY_MODES
}


def common_frame_config(max_rotation_deg: float | None, *, probability: float = COMMON_BODY_FRAME_PROBABILITY) -> CommonBodyFrameRandomizationConfig:
    rotation = None if max_rotation_deg is None else PerturbationMagnitudeConfig(maximum=np.deg2rad(max_rotation_deg), distribution=PerturbationDistribution.UNIFORM)
    return CommonBodyFrameRandomizationConfig(enabled=True, probability=probability, rotation_axes=COMMON_BODY_FRAME_ROTATION_AXES, rotation=rotation)


# Both samplers are FIXED for this experiment, unlike notebook 39's yaw curriculum.
frame_view_randomizer = CommonBodyFrameRandomizer(common_frame_config(COMMON_BODY_FRAME_MAX_ROTATION_DEG))
full_range_frame_view_randomizer = CommonBodyFrameRandomizer(common_frame_config(COMMON_BODY_FRAME_MAX_ROTATION_DEG))
print("Topology pipelines:", {name: pipe.config.sensor_topology for name, pipe in context_augmentation_pipelines.items()})
print("Single-fault prior:", pair_prior_perturber.config)
print("Fixed common-body frame:", frame_view_randomizer.config)

## 5. Pooled model and complete window pipeline (architecture and weights unchanged)

In [ ]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation="gelu"),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation="gelu", dropout=MLP_DROPOUT),
    pooler=CalibrationPoolerConfig(
        n_heads=POOLER_N_HEADS,
        dropout=POOLER_DROPOUT,
        calibration_context_dim=0,           # CALIBRATION_CONTEXT_DIM
    ),

    head=PooledCalibrationHeadConfig(
        calibration_context_dim=0,
        hidden_dims=HEAD_HIDDEN_DIMS,
        activation="gelu",
        dropout=HEAD_DROPOUT,
    ),
)

geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibrationModel(model_config).to(device=DEVICE, dtype=MODEL_DTYPE)
calibration_updater = CalibrationUpdater()
window_step = WindowStep(geometry_processor=geometry_processor, tokenizer=tokenizer, model=model, calibration_updater=calibration_updater)

# if not getattr(model, "uses_calibration_context", False):
#     raise RuntimeError("This experiment requires the pooled model to use numeric calibration context.")

print(model)
print()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
print("Pooler calibration context dim:", model.pooler.calibration_context_dim)
print("Head calibration context dim:", model.head.config.calibration_context_dim)


## 6. Dynamic-key three-view pipeline and per-window topology grouping

Every view in a group uses the same physical topology, measurements and true calibration. The original and opposite-prior views are paired around exactly one perturbed sensor. Calibration-key ordering comes from the actual window, rather than the fixed KAIST tuple. This is required for `imu_synth`.

In [ ]:
def make_augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(seed)


def clone_window_with_calibration(window: WindowBatch, calibration):
    return WindowBatch(streams=dict(window.streams), current_calibration=calibration, metadata=dict(window.metadata), targets=None if window.targets is None else dict(window.targets), dataset_indices=window.dataset_indices)


def make_equivariant_windows(base_window: WindowBatch, *, generator: torch.Generator, frame_probability: float | None = None, return_negative: bool = False, frame_randomizer: CommonBodyFrameRandomizer | None = None, physical_pipeline: AugmentationPipeline | None = None):
    """One set of physical samples and one signed calibration prior. Return 2 views, optionally a third opposite-prior view."""
    context_result = (context_augmentation_pipeline if physical_pipeline is None else physical_pipeline)(base_window, generator=generator)
    physical_window = context_result.augmented_window
    true_calibration = physical_window.current_calibration
    _, delta_xi, delta_tau = pair_prior_perturber(true_calibration, generator=generator)
    candidate_calibration = apply_prior_perturbation(true_calibration, delta_xi, delta_tau)
    original_window = clone_window_with_calibration(physical_window, candidate_calibration)
    # Training uses the per-epoch bounded sampler. Validation/test pass the fixed full selected-axis sampler.
    augmenter = frame_view_randomizer if frame_randomizer is None else frame_randomizer
    if frame_probability is not None:
        # Preserve the chosen angle bound when only overriding the probability.
        augmenter = CommonBodyFrameRandomizer(CommonBodyFrameRandomizationConfig(enabled=True, probability=frame_probability, rotation_axes=augmenter.config.rotation_axes, rotation=augmenter.config.rotation))
    rotated_window, _, H = augmenter(original_window, generator=generator)
    active = torch.linalg.vector_norm(H[:, :3, :3] - torch.eye(3, dtype=H.dtype, device=H.device), dim=(1, 2)) > 1e-5
    if not return_negative:
        return original_window, rotated_window, H, active
    opposite_calibration = apply_prior_perturbation(true_calibration, {key: -value for key, value in delta_xi.items()}, {key: -value for key, value in delta_tau.items()})
    opposite_window = clone_window_with_calibration(physical_window, opposite_calibration)
    return original_window, rotated_window, H, active, opposite_window


def correction_target(window: WindowBatch, calibration_key: str):
    if window.targets is None or calibration_key not in window.targets:
        raise ValueError(f"Missing calibration target for {calibration_key!r}.")
    target = window.targets[calibration_key]
    current = window.current_calibration[calibration_key]
    if target.next_transform is None or target.next_time_offset is None:
        raise ValueError(f"Incomplete transform/clock target for {calibration_key!r}.")
    return se3_log(target.next_transform @ torch.linalg.inv(current.transform)), target.next_time_offset - current.time_offset


def rotate_twist(delta_xi: torch.Tensor, G: torch.Tensor) -> torch.Tensor:
    """Adjoint action for H=[G,0;0,1] and twist ordering [rotation, translation]."""
    return torch.cat((torch.bmm(G, delta_xi[:, :3, None]).squeeze(-1), torch.bmm(G, delta_xi[:, 3:, None]).squeeze(-1)), dim=-1)


# A WindowBatch has a single stream schema. Per-window topology draws therefore
# produce 1-3 homogeneous groups INSIDE the original DataLoader minibatch.
def topology_slice_batch(window: WindowBatch, selected_indices: torch.Tensor) -> WindowBatch:
    batch_size = next(iter(window.current_calibration.values())).transform.shape[0]
    if selected_indices.numel() == 0:
        raise ValueError("Cannot construct an empty topology group")

    def select(obj):
        if isinstance(obj, torch.Tensor) and obj.ndim > 0 and obj.shape[0] == batch_size:
            return obj.index_select(0, selected_indices.to(device=obj.device))
        if is_dataclass(obj) and not isinstance(obj, type):
            updates = {field.name: select(getattr(obj, field.name)) for field in fields(obj) if field.init}
            return replace(obj, **updates)
        if isinstance(obj, dict):
            return {key: select(value) for key, value in obj.items()}
        return obj

    sliced = select(window)
    if sliced.dataset_indices is not None and not isinstance(sliced.dataset_indices, torch.Tensor):
        if len(sliced.dataset_indices) == batch_size:
            indices_cpu = selected_indices.detach().cpu().tolist()
            selected = [sliced.dataset_indices[j] for j in indices_cpu]
            sliced = replace(sliced, dataset_indices=type(sliced.dataset_indices)(selected))
    return sliced


def topology_groups(window: WindowBatch, generator: torch.Generator, mode: str = "mixture"):
    batch_size = next(iter(window.current_calibration.values())).transform.shape[0]
    if mode == "mixture":
        draw = torch.rand((batch_size,), generator=generator, device=next(iter(window.current_calibration.values())).transform.device)
        assigned = torch.where(draw < TOPOLOGY_ORIGINAL_PROBABILITY, 0, torch.where(draw < TOPOLOGY_ORIGINAL_PROBABILITY + TOPOLOGY_REPLACE_LEFT_PROBABILITY, 1, 2))
    elif mode in TOPOLOGY_MODES:
        assigned = torch.full((batch_size,), TOPOLOGY_MODES.index(mode), device=next(iter(window.current_calibration.values())).transform.device, dtype=torch.long)
    else:
        raise ValueError(f"Unknown topology evaluation mode {mode!r}")
    return [(topology, topology_slice_batch(window, positions)) for value, topology in enumerate(TOPOLOGY_MODES) if (positions := torch.nonzero(assigned == value, as_tuple=False).flatten()).numel()]


def topology_window_step(window: WindowBatch, calibration=None):
    """Original WindowStep call, with the tokenizer adapted to actual active keys."""
    states = window.current_calibration if calibration is None else calibration
    step = WindowStep(geometry_processor=geometry_processor, tokenizer=Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=tuple(states)), model=model, calibration_updater=calibration_updater)
    return step(window, calibration=calibration) if calibration is not None else step(window)


def topology_mode_of(window: WindowBatch) -> str:
    keys = set(window.current_calibration)
    if keys == set(CALIBRATION_KEYS): return "original"
    if keys == {"imu", "imu_synth", "lidar_right"}: return "replace_lidar"
    if keys == {"imu", "lidar", "imu_synth"}: return "replace_lidar_right"
    raise RuntimeError(f"Unexpected active calibration keys {keys}")

## 7. Physical topology and supervision semantic checks

In [ ]:
semantic_base_window = window_batch_to(next(iter(train_loader)), DEVICE, dtype=MODEL_DTYPE)
semantic_original, semantic_rotated, semantic_H, semantic_active = make_equivariant_windows(semantic_base_window, generator=make_augmentation_generator(TRAIN_PAIR_SEED + 1), frame_probability=1.0, physical_pipeline=context_augmentation_pipelines["replace_lidar"])
G = semantic_H[:, :3, :3]
assert bool(semantic_active.all()), "The semantic check must rotate every example."
for stream_key in semantic_original.streams:
    a, b = semantic_original.streams[stream_key], semantic_rotated.streams[stream_key]
    torch.testing.assert_close(a.values, b.values, rtol=0, atol=0)
    torch.testing.assert_close(a.timestamps, b.timestamps, rtol=0, atol=0)
    if a.interval_start_timestamps is not None:
        torch.testing.assert_close(a.interval_start_timestamps, b.interval_start_timestamps, rtol=0, atol=0)
for key in semantic_original.current_calibration:
    a, b = semantic_original.current_calibration[key], semantic_rotated.current_calibration[key]
    torch.testing.assert_close(b.transform, semantic_H @ a.transform, rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(b.time_offset, a.time_offset, rtol=0, atol=0)
    orig_xi, orig_tau = correction_target(semantic_original, key)
    rot_xi, rot_tau = correction_target(semantic_rotated, key)
    torch.testing.assert_close(rot_xi, rotate_twist(orig_xi, G), rtol=5e-4, atol=5e-5)
    torch.testing.assert_close(rot_tau, orig_tau, rtol=0, atol=1e-7)
    for other in semantic_original.current_calibration:
        rel_original = torch.linalg.inv(a.transform) @ semantic_original.current_calibration[other].transform
        rel_rotated = torch.linalg.inv(b.transform) @ semantic_rotated.current_calibration[other].transform
        torch.testing.assert_close(rel_original, rel_rotated, rtol=1e-5, atol=1e-5)
# Verify the first curriculum stage in the actual WindowBatch semantics test.
if frame_view_randomizer.config.rotation is not None:
    trace = semantic_H[:, :3, :3].diagonal(dim1=-2, dim2=-1).sum(-1)
    sampled_angles_deg = torch.rad2deg(torch.acos(((trace - 1.0) / 2.0).clamp(-1.0, 1.0)))
    cap_deg = float(np.rad2deg(frame_view_randomizer.config.rotation.maximum))
    assert bool((sampled_angles_deg <= cap_deg + 0.05).all()), "Common-frame rotation exceeded the configured cap."
    print(f"PASS: initial bounded frame angles <= {cap_deg:.1f} deg; maximum observed {sampled_angles_deg.max().item():.3f} deg")
print("PASS: shared frame, identical raw sensor measurements, fixed relative extrinsics, adjoint-consistent supervision.")
assert topology_mode_of(semantic_original) == "replace_lidar"
assert topology_mode_of(semantic_plus) == "replace_lidar_right"
print("PASS: both synthetic-IMU replacement directions preserve original target construction.")
print("Median paired frame rotation [deg]:", torch.rad2deg(torch.linalg.vector_norm(se3_log(semantic_H)[:, :3], dim=-1)).median().item())

# Check exact opposite-prior correction targets, including the time offset, without assuming small angles.
semantic_plus, _, _, _, semantic_minus = make_equivariant_windows(window_batch_to(next(iter(train_loader)), DEVICE, dtype=MODEL_DTYPE), generator=make_augmentation_generator(TRAIN_PAIR_SEED + 2), frame_probability=1.0, return_negative=True, physical_pipeline=context_augmentation_pipelines["replace_lidar_right"])
for key in semantic_plus.current_calibration:
    xi_plus, tau_plus = correction_target(semantic_plus, key)
    xi_minus, tau_minus = correction_target(semantic_minus, key)
    torch.testing.assert_close(xi_plus, -xi_minus, rtol=1e-3, atol=2e-4)
    torch.testing.assert_close(tau_plus, -tau_minus, rtol=1e-4, atol=2e-6)
    for stream_key in semantic_plus.streams:
        torch.testing.assert_close(semantic_plus.streams[stream_key].values, semantic_minus.streams[stream_key].values, rtol=0, atol=0)
print("PASS: positive/negative prior targets are opposite and share the physical measurements.")
del semantic_plus, semantic_minus

del semantic_base_window, semantic_original, semantic_rotated, semantic_H, semantic_active, G
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 8. Supervised correction, active-fault emphasis, equivariance, and optional pair-difference objective

**Why the cross-frame signed pair differs from merely forcing opposite predictions:** `L_pair` compares predicted signed **differences** with differences of the exact supervised correction targets. Zero predictions on an active fault therefore incur a positive pair loss. This information is algebraically implied by the two supervised targets; it is a regularizer/gradient reweighting, not new LiDAR or IMU information.

The additional active-rotation term is the primary intervention for the observed line `rotated error ≈ target magnitude`, because `one_calibration_per_sample=True` leaves two calibration heads with zero targets. A zero-output baseline must remain visible in the held-out metrics.


**39b changes:** active rotation supervision receives coefficient 0.75, with 2× emphasis on rotated-frame active faults. The `pair` term is now the **cross-frame signed ROTATION gap** (0.10), using rotated-positive and transformed original-negative predictions. Equivariance weight remains 0.05 to avoid rewarding two jointly shrunken predictions. The `pair` metric no longer includes temporal or translational differences; those remain supervised by the regular losses.


In [ ]:
def correction_regression_loss(prediction, target):
    if CORRECTION_LOSS_TYPE == "mse": return F.mse_loss(prediction, target)
    if CORRECTION_LOSS_TYPE == "smooth_l1": return F.smooth_l1_loss(prediction, target, beta=1.0)
    if CORRECTION_LOSS_TYPE == "l1": return F.l1_loss(prediction, target)
    raise ValueError(f"Unknown CORRECTION_LOSS_TYPE={CORRECTION_LOSS_TYPE!r}.")


def compute_training_loss(window, output):
    total = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
    metrics = {}
    active_rotation_errors = []
    active_tau_errors = []
    keys = tuple(window.current_calibration)
    for key in keys:
        pred = output.predictions[key]
        target_xi, target_tau = correction_target(window, key)
        lr = correction_regression_loss(pred.delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD, target_xi[:, :3] / ROTATION_LOSS_SCALE_RAD)
        lt = correction_regression_loss(pred.delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M, target_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M)
        l_tau = correction_regression_loss(pred.delta_tau / TIME_OFFSET_LOSS_SCALE_S, target_tau / TIME_OFFSET_LOSS_SCALE_S)
        total = total + ROTATION_LOSS_WEIGHT * lr + TRANSLATION_LOSS_WEIGHT * lt + TIME_OFFSET_LOSS_WEIGHT * l_tau
        metrics[f"{key}/rotation_error_deg"] = torch.rad2deg(torch.linalg.vector_norm(pred.delta_xi[:, :3] - target_xi[:, :3], dim=-1)).mean().item()
        metrics[f"{key}/time_offset_error_ms"] = (1000.0 * torch.abs(pred.delta_tau - target_tau)).mean().item()
        metrics[f"{key}/translation_error_cm"] = (100.0 * torch.linalg.vector_norm(pred.delta_xi[:, 3:] - target_xi[:, 3:], dim=-1)).mean().item()
        rotation_active = torch.linalg.vector_norm(target_xi[:, :3], dim=-1) > 1e-6
        tau_active = target_tau[:, 0].abs() > 1e-7
        metrics[f"{key}/rotation_active_fraction"] = rotation_active.float().mean().item()
        metrics[f"{key}/tau_active_fraction"] = tau_active.float().mean().item()
        if bool(rotation_active.any()): active_rotation_errors.append(torch.rad2deg(torch.linalg.vector_norm(pred.delta_xi[rotation_active, :3] - target_xi[rotation_active, :3], dim=-1)))
        if bool(tau_active.any()): active_tau_errors.append(1000.0 * (pred.delta_tau[tau_active] - target_tau[tau_active]).abs().reshape(-1))
    total = total / len(keys)
    # These are evaluated ONLY where the calibration target really requires a correction.
    # They distinguish learning the injected errors from merely predicting zero for healthy sensors.
    metrics["active_rotation_error_deg"] = torch.cat(active_rotation_errors).mean().item() if active_rotation_errors else 0.0
    metrics["active_tau_error_ms"] = torch.cat(active_tau_errors).mean().item() if active_tau_errors else 0.0
    metrics["active_rotation_count"] = sum(len(x) for x in active_rotation_errors)
    metrics["active_tau_count"] = sum(len(x) for x in active_tau_errors)
    return total, metrics


def compute_equivariance_loss(original_output, rotated_output, G, active):
    losses = []
    metrics = {"active_fraction": active.float().mean().item()}
    keys = tuple(original_output.predictions)
    for key in keys:
        pa = original_output.predictions[key]
        pb = rotated_output.predictions[key]
        expected_xi = rotate_twist(pa.delta_xi, G)
        if bool(active.any()):
            r_err = (pb.delta_xi[active, :3] - expected_xi[active, :3]) / ROTATION_LOSS_SCALE_RAD
            t_err = (pb.delta_xi[active, 3:] - expected_xi[active, 3:]) / TRANSLATION_LOSS_SCALE_M
            tau_err = (pb.delta_tau[active] - pa.delta_tau[active]) / TIME_OFFSET_LOSS_SCALE_S
            rotation_loss = r_err.square().mean()
            translation_loss = t_err.square().mean()
            time_loss = tau_err.square().mean()
            metrics[f"{key}/rotation_equivariance_deg"] = torch.rad2deg(torch.linalg.vector_norm(r_err * ROTATION_LOSS_SCALE_RAD, dim=-1)).mean().item()
            metrics[f"{key}/tau_equivariance_ms"] = (1000.0 * tau_err.abs() * TIME_OFFSET_LOSS_SCALE_S).mean().item()
        else:
            zero = (pa.delta_xi.sum() + pb.delta_xi.sum() + pa.delta_tau.sum() + pb.delta_tau.sum()) * 0.0
            rotation_loss = translation_loss = time_loss = zero
            metrics[f"{key}/rotation_equivariance_deg"] = 0.0
            metrics[f"{key}/tau_equivariance_ms"] = 0.0
        losses.append(ROTATION_LOSS_WEIGHT * rotation_loss + TRANSLATION_LOSS_WEIGHT * translation_loss + TIME_OFFSET_LOSS_WEIGHT * time_loss)
        metrics[f"{key}/loss_rotation"] = rotation_loss.detach().item()
        metrics[f"{key}/loss_translation"] = translation_loss.detach().item()
        metrics[f"{key}/loss_time_offset"] = time_loss.detach().item()
    return torch.stack(losses).mean(), metrics



def compute_active_rotation_supervision(window, output):
    """Additional MSE only on *injected* rotation targets, aggregated over active keys and examples."""
    discrepancies = []
    predicted_norms = []
    target_norms = []
    keys = tuple(window.current_calibration)
    for key in keys:
        target_xi, _ = correction_target(window, key)
        target_phi = target_xi[:, :3]
        pred_phi = output.predictions[key].delta_xi[:, :3]
        mask = torch.linalg.vector_norm(target_phi, dim=-1) > 1e-6
        if bool(mask.any()):
            discrepancies.append(((pred_phi[mask] - target_phi[mask]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            predicted_norms.append(torch.rad2deg(torch.linalg.vector_norm(pred_phi[mask], dim=-1)))
            target_norms.append(torch.rad2deg(torch.linalg.vector_norm(target_phi[mask], dim=-1)))
    if not discrepancies:
        first = output.predictions[keys[0]].delta_xi
        return first.sum() * 0.0, {"active_rotation_target_deg": 0.0, "active_rotation_prediction_deg": 0.0}
    rotation_loss = ROTATION_LOSS_WEIGHT * torch.cat(discrepancies).mean()
    return rotation_loss, {"active_rotation_target_deg": torch.cat(target_norms).mean().item(), "active_rotation_prediction_deg": torch.cat(predicted_norms).mean().item()}


def compute_signed_pair_difference_loss(positive_window, positive_output, negative_window, negative_output):
    """Supervised difference loss for the same physical samples under +/- errors; only selected fault components contribute."""
    rotation_errors, translation_errors, time_errors = [], [], []
    target_differences, predicted_differences = [], []
    keys = tuple(positive_window.current_calibration)
    for key in keys:
        pos_target_xi, pos_target_tau = correction_target(positive_window, key)
        neg_target_xi, neg_target_tau = correction_target(negative_window, key)
        pos_prediction = positive_output.predictions[key]
        neg_prediction = negative_output.predictions[key]
        target_diff_xi = pos_target_xi - neg_target_xi
        predicted_diff_xi = pos_prediction.delta_xi - neg_prediction.delta_xi
        target_diff_tau = pos_target_tau - neg_target_tau
        predicted_diff_tau = pos_prediction.delta_tau - neg_prediction.delta_tau
        rotation_mask = torch.linalg.vector_norm(target_diff_xi[:, :3], dim=-1) > 1e-6
        translation_mask = torch.linalg.vector_norm(target_diff_xi[:, 3:], dim=-1) > 1e-6
        time_mask = target_diff_tau[:, 0].abs() > 1e-7
        if bool(rotation_mask.any()):
            rotation_errors.append(((predicted_diff_xi[rotation_mask, :3] - target_diff_xi[rotation_mask, :3]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            target_differences.append(torch.linalg.vector_norm(target_diff_xi[rotation_mask, :3], dim=-1))
            predicted_differences.append(torch.linalg.vector_norm(predicted_diff_xi[rotation_mask, :3], dim=-1))
        if bool(translation_mask.any()):
            translation_errors.append(((predicted_diff_xi[translation_mask, 3:] - target_diff_xi[translation_mask, 3:]) / TRANSLATION_LOSS_SCALE_M).square().mean(dim=-1))
        if bool(time_mask.any()):
            time_errors.append(((predicted_diff_tau[time_mask, 0] - target_diff_tau[time_mask, 0]) / TIME_OFFSET_LOSS_SCALE_S).square())
    zero = positive_output.predictions[keys[0]].delta_xi.sum() * 0.0
    lr = torch.cat(rotation_errors).mean() if rotation_errors else zero
    lt = torch.cat(translation_errors).mean() if translation_errors else zero
    l_tau = torch.cat(time_errors).mean() if time_errors else zero
    pair_loss = ROTATION_LOSS_WEIGHT * lr + TRANSLATION_LOSS_WEIGHT * lt + TIME_OFFSET_LOSS_WEIGHT * l_tau
    metrics = {"loss_rotation": lr.detach().item(), "loss_translation": lt.detach().item(), "loss_time_offset": l_tau.detach().item(), "active_rotation_pairs": sum(x.numel() for x in target_differences)}
    if target_differences:
        metrics["target_rotation_gap_deg"] = torch.rad2deg(torch.cat(target_differences)).mean().item()
        metrics["predicted_rotation_gap_deg"] = torch.rad2deg(torch.cat(predicted_differences)).mean().item()
    else:
        metrics["target_rotation_gap_deg"] = metrics["predicted_rotation_gap_deg"] = 0.0
    return pair_loss, metrics


def compute_cross_frame_signed_rotation_loss(rotated_window, rotated_output, opposite_window, opposite_output, G):
    """Learn the signed calibration response *in the rotated frame*, without a fourth forward pass.

    The opposite-prior prediction (original coordinates) and its exact target are
    both transformed with G. The difference must match the actual difference
    between the two correction targets. All-zero predictions incur a positive
    penalty whenever a nonzero rotation fault is present.
    """
    rotation_errors, predicted_gaps, true_gaps = [], [], []
    keys = tuple(rotated_window.current_calibration)
    for key in keys:
        rotated_xi, _ = correction_target(rotated_window, key)
        opposite_xi, _ = correction_target(opposite_window, key)
        true_gap = rotated_xi[:, :3] - torch.bmm(G, opposite_xi[:, :3, None]).squeeze(-1)
        pred_rotated = rotated_output.predictions[key].delta_xi[:, :3]
        pred_opposite = opposite_output.predictions[key].delta_xi[:, :3]
        predicted_gap = pred_rotated - torch.bmm(G, pred_opposite[:, :, None]).squeeze(-1)
        fault_mask = torch.linalg.vector_norm(true_gap, dim=-1) > 1e-6
        if bool(fault_mask.any()):
            rotation_errors.append(((predicted_gap[fault_mask] - true_gap[fault_mask]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            predicted_gaps.append(torch.rad2deg(torch.linalg.vector_norm(predicted_gap[fault_mask], dim=-1)))
            true_gaps.append(torch.rad2deg(torch.linalg.vector_norm(true_gap[fault_mask], dim=-1)))
    if not rotation_errors:
        zero = rotated_output.predictions[keys[0]].delta_xi.sum() * 0.0
        return zero, {"target_rotation_gap_deg": 0.0, "predicted_rotation_gap_deg": 0.0, "active_rotation_pairs": 0}
    loss = ROTATION_LOSS_WEIGHT * torch.cat(rotation_errors).mean()
    return loss, {
        "target_rotation_gap_deg": torch.cat(true_gaps).mean().item(),
        "predicted_rotation_gap_deg": torch.cat(predicted_gaps).mean().item(),
        "active_rotation_pairs": sum(x.numel() for x in true_gaps),
    }


def compute_equivariant_objective(original_window, original_output, rotated_window, rotated_output, common_frame, active, opposite_window=None, opposite_output=None):
    if (opposite_window is None) != (opposite_output is None):
        raise ValueError("opposite_window and opposite_output must both be supplied or both be omitted.")
    original_loss, original_metrics = compute_training_loss(original_window, original_output)
    rotated_loss, rotated_metrics = compute_training_loss(rotated_window, rotated_output)
    original_active_loss, original_active_metrics = compute_active_rotation_supervision(original_window, original_output)
    rotated_active_loss, rotated_active_metrics = compute_active_rotation_supervision(rotated_window, rotated_output)
    equiv_loss, equiv_metrics = compute_equivariance_loss(original_output, rotated_output, common_frame[:, :3, :3], active)
    view_weight = 2.0
    supervision_sum = original_loss + rotated_loss
    active_sum = original_active_loss + ROTATED_ACTIVE_MULTIPLIER * rotated_active_loss
    active_view_weight = 1.0 + ROTATED_ACTIVE_MULTIPLIER
    pair_loss = original_loss * 0.0
    pair_metrics = {}
    opposite_metrics = {}
    if opposite_window is not None:
        opposite_loss, opposite_metrics = compute_training_loss(opposite_window, opposite_output)
        opposite_active_loss, opposite_active_metrics = compute_active_rotation_supervision(opposite_window, opposite_output)
        supervision_sum = supervision_sum + OPPOSITE_VIEW_SUPERVISION_WEIGHT * opposite_loss
        active_sum = active_sum + OPPOSITE_VIEW_SUPERVISION_WEIGHT * opposite_active_loss
        active_view_weight += OPPOSITE_VIEW_SUPERVISION_WEIGHT
        view_weight += OPPOSITE_VIEW_SUPERVISION_WEIGHT
        pair_loss, pair_metrics = compute_cross_frame_signed_rotation_loss(rotated_window, rotated_output, opposite_window, opposite_output, common_frame[:, :3, :3])
        opposite_metrics.update(opposite_active_metrics)
    elif PAIR_LOSS_WEIGHT > 0.0:
        raise ValueError("Nonzero PAIR_LOSS_WEIGHT requires the opposite-prior view.")
    supervised = supervision_sum / view_weight
    active_supervised = active_sum / active_view_weight
    supervised_selection = supervised + ACTIVE_ROTATION_SUPERVISION_WEIGHT * active_supervised
    loss = supervised_selection + EQUIVARIANCE_LOSS_WEIGHT * equiv_loss + PAIR_LOSS_WEIGHT * pair_loss
    metrics = {
        "loss_total": loss.detach().item(), "loss_regular": 0.5 * (original_loss + rotated_loss).detach().item(),
        "loss_supervised": supervised.detach().item(), "loss_active_supervised": active_supervised.detach().item(),
        "loss_supervised_selection": supervised_selection.detach().item(), "loss_equivariance": equiv_loss.detach().item(),
        "loss_pair": pair_loss.detach().item(), "loss_original": original_loss.detach().item(), "loss_rotated": rotated_loss.detach().item(),
        "weighted_active": ACTIVE_ROTATION_SUPERVISION_WEIGHT * active_supervised.detach().item(),
        "weighted_equivariance": EQUIVARIANCE_LOSS_WEIGHT * equiv_loss.detach().item(),
        "weighted_pair": PAIR_LOSS_WEIGHT * pair_loss.detach().item(),
    }
    metrics.update({f"original/{k}": v for k, v in original_metrics.items()})
    metrics.update({f"rotated/{k}": v for k, v in rotated_metrics.items()})
    metrics.update({f"equiv/{k}": v for k, v in equiv_metrics.items()})
    metrics.update({f"pair/{k}": v for k, v in pair_metrics.items()})
    metrics.update({f"opposite/{k}": v for k, v in opposite_metrics.items()})
    metrics.update({f"original/{k}": v for k, v in original_active_metrics.items()})
    metrics.update({f"rotated/{k}": v for k, v in rotated_active_metrics.items()})
    return loss, metrics


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=COSINE_MIN_LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ConstantLR(optimizer, factor=1.0, total_iters=NUM_EPOCHS)

## 9. Three-view gradient and oracle checks across all topologies

In [ ]:
from types import SimpleNamespace

smoke_base = window_batch_to(collate_windows([train_dataset[i] for i in (0, min(len(train_dataset) - 1, 100))]), DEVICE, dtype=MODEL_DTYPE)
# Axis-mask smoke test. The shared frame must be a pure Z-axis yaw for every batch item.
_yaw_smoke = CommonBodyFrameRandomizer(common_frame_config(180.0, probability=1.0))
_, _, _H_yaw = _yaw_smoke(smoke_base, generator=make_augmentation_generator(TRAIN_PAIR_SEED + 1234))
assert torch.allclose(_H_yaw[:, 2, :3], torch.tensor([0.0, 0.0, 1.0], device=DEVICE, dtype=MODEL_DTYPE).expand(_H_yaw.shape[0], 3), atol=1e-5)
assert torch.allclose(_H_yaw[:, :3, 2], torch.tensor([0.0, 0.0, 1.0], device=DEVICE, dtype=MODEL_DTYPE).expand(_H_yaw.shape[0], 3), atol=1e-5)
print('Yaw-only shared-frame smoke: PASS')

smoke_original, smoke_rotated, smoke_H, smoke_active, smoke_opposite = make_equivariant_windows(smoke_base, generator=make_augmentation_generator(TRAIN_PAIR_SEED + 22), frame_probability=1.0, return_negative=True, physical_pipeline=context_augmentation_pipelines["replace_lidar_right"])

# Algebraic oracle checks for the opposite-signed *target differences*, independent of network initialization.
def _dummy_calibration_output(window, *, predict_zero: bool):
    predictions = {}
    for key in window.current_calibration:
        true_xi, true_tau = correction_target(window, key)
        predictions[key] = SimpleNamespace(delta_xi=torch.zeros_like(true_xi) if predict_zero else true_xi, delta_tau=torch.zeros_like(true_tau) if predict_zero else true_tau)
    return SimpleNamespace(predictions=predictions)

oracle_pair, _ = compute_signed_pair_difference_loss(smoke_original, _dummy_calibration_output(smoke_original, predict_zero=False), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=False))
zero_pair, _ = compute_signed_pair_difference_loss(smoke_original, _dummy_calibration_output(smoke_original, predict_zero=True), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=True))
assert abs(oracle_pair.item()) < 1e-8, f"Oracle pair loss should be zero: {oracle_pair.item()}"
assert zero_pair.item() > 0.1, "All-zero outputs must NOT solve supervised signed-pair loss."
print(f"Pair oracle: {oracle_pair.item():.7f} | all-zero baseline: {zero_pair.item():.3f}")
oracle_cross, _ = compute_cross_frame_signed_rotation_loss(smoke_rotated, _dummy_calibration_output(smoke_rotated, predict_zero=False), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=False), smoke_H[:, :3, :3])
zero_cross, _ = compute_cross_frame_signed_rotation_loss(smoke_rotated, _dummy_calibration_output(smoke_rotated, predict_zero=True), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=True), smoke_H[:, :3, :3])
assert abs(oracle_cross.item()) < 1e-6, f"Cross-frame oracle loss should be zero: {oracle_cross.item()}"
assert zero_cross.item() > 0.1, "All-zero outputs must NOT solve the cross-frame signed rotation loss."
print(f"Cross-frame signed oracle: {oracle_cross.item():.7f} | all-zero penalty: {zero_cross.item():.3f}")

model.train()
optimizer.zero_grad(set_to_none=True)
smoke_original_out = topology_window_step(smoke_original).model_output
smoke_rotated_out = topology_window_step(smoke_rotated).model_output
smoke_opposite_out = topology_window_step(smoke_opposite).model_output if USE_SIGNED_PAIR else None
smoke_loss, smoke_metrics = compute_equivariant_objective(smoke_original, smoke_original_out, smoke_rotated, smoke_rotated_out, smoke_H, smoke_active, smoke_opposite if USE_SIGNED_PAIR else None, smoke_opposite_out)
smoke_loss.backward()
grad_norm = sum(float(p.grad.detach().square().sum().cpu()) for p in model.parameters() if p.grad is not None) ** 0.5
print("Smoke objective:", {k: round(v, 4) for k, v in smoke_metrics.items() if k.startswith('loss_') or k.startswith('weighted_')}, "gradient norm:", grad_norm)
assert np.isfinite(smoke_loss.item()) and np.isfinite(grad_norm) and grad_norm > 0
assert smoke_metrics["equiv/active_fraction"] > 0.99
assert "imu_synth" in smoke_original.current_calibration
print("THREE-VIEW GRADIENT SMOKE: PASS")
optimizer.zero_grad(set_to_none=True)
del smoke_original, smoke_rotated, smoke_opposite, smoke_H, smoke_active, smoke_original_out, smoke_rotated_out, smoke_opposite_out, smoke_loss, smoke_metrics, oracle_pair, zero_pair, oracle_cross, zero_cross
if DEVICE.type == "cuda": torch.cuda.empty_cache()
# Confirm actual per-window sampling and both synthetic mounting geometries.
counts = {mode: 0 for mode in TOPOLOGY_MODES}
sampling_generator = make_augmentation_generator(823456)
for _ in range(100):
    for mode, subset in topology_groups(smoke_base, sampling_generator, mode="mixture"):
        counts[mode] += next(iter(subset.current_calibration.values())).transform.shape[0]
assert all(counts[key] > 0 for key in TOPOLOGY_MODES), counts
assert abs(counts["original"] / sum(counts.values()) - TOPOLOGY_ORIGINAL_PROBABILITY) < 0.10, counts
print("PASS: per-window 50/25/25 categorical sampling; counts:", counts)
for mode in TOPOLOGY_MODES:
    context = context_augmentation_pipelines[mode](smoke_base, generator=make_augmentation_generator(7123))
    output = context.augmented_window
    assert topology_mode_of(output) == mode
    for key in output.current_calibration:
        assert key in output.targets and output.targets[key].next_transform is not None
print("PASS: physical pipeline/targets for all three topologies")
# Geometric oracle for the SYNTHETIC IMU before true-calibration augmentation.
for mode in ("replace_lidar", "replace_lidar_right"):
    raw_topology, choice = context_augmentation_pipelines[mode].sensor_topology_augmenter(smoke_base, generator=make_augmentation_generator(99800))
    assert choice == mode and len(raw_topology.current_calibration) == 3
    R_original = raw_topology.current_calibration["imu"].transform[:, :3, :3]
    R_second = raw_topology.current_calibration["imu_synth"].transform[:, :3, :3]
    for native, synthetic in (("gyro", "gyro_synth"), ("accel", "accel_synth")):
        x_native = raw_topology.streams[native].values @ R_original.transpose(-1, -2)
        x_synthetic = raw_topology.streams[synthetic].values @ R_second.transpose(-1, -2)
        torch.testing.assert_close(x_native, x_synthetic, rtol=2e-5, atol=2e-5)
        torch.testing.assert_close(raw_topology.streams[native].timestamps, raw_topology.streams[synthetic].timestamps)
        assert raw_topology.metadata[synthetic].calibration_key == "imu_synth"
    torch.testing.assert_close(raw_topology.current_calibration["imu"].time_offset, raw_topology.current_calibration["imu_synth"].time_offset)
print("PASS: both synthetic IMU mountings produce identical true body motion with unmodified source IMU.")

del smoke_base

## 10. Optional stochastic precheck, separated by physical topology

This **does not change the full-run weights or optimizer**. It trains briefly on randomly mixed per-window topologies, while evaluating every 25 steps on three **fixed, seeded** tasks: original KAIST 1I2L, replace left LiDAR, and replace right LiDAR. At step 0 it also evaluates the untouched 002.23 checkpoint. Compare **active rotation gain and tau error**, not only total training loss. A constant (usually 0%) common-frame yaw is used for the isolated stochastic check; the full run uses its separate fixed 25% yaw rate.

In [ ]:
# A NEW stochastic test for the topology mixture, not the previous Z-yaw curriculum.
RUN_STOCHASTIC_PRECHECK = True
STOCHASTIC_PRECHECK_STEPS = 250
PRECHECK_PRINT_EVERY = 25
PRECHECK_EVAL_SEED = VALIDATION_PAIR_SEED + 200000
PRECHECK_COMMON_YAW_PROBABILITY = 0.0  # First isolate topology; use 0.25 in full training.


def precheck_metrics_on_mode(base_window, mode, seed, sampler):
    generator = make_augmentation_generator(seed)
    pipeline = context_augmentation_pipelines[mode]
    original, rotated, H, active, opposite = make_equivariant_windows(base_window, generator=generator, frame_randomizer=sampler, physical_pipeline=pipeline, return_negative=True)
    output_a = topology_window_step(original).model_output
    output_b = topology_window_step(rotated).model_output
    output_c = topology_window_step(opposite).model_output if USE_SIGNED_PAIR else None
    _, metrics = compute_equivariant_objective(original, output_a, rotated, output_b, H, active, opposite if USE_SIGNED_PAIR else None, output_c)
    # Same positive-prior corrected physical problem is compared at every report.
    gains, tau_errors = [], []
    for key in original.current_calibration:
        target_xi, target_tau = correction_target(original, key)
        target_phi = target_xi[:, :3]
        predicted_phi = output_a.predictions[key].delta_xi[:, :3]
        active_fault = torch.linalg.vector_norm(target_phi, dim=-1) > np.deg2rad(1.0)
        if bool(active_fault.any()):
            target_norm = torch.linalg.vector_norm(target_phi[active_fault], dim=-1)
            residual = torch.linalg.vector_norm((predicted_phi - target_phi)[active_fault], dim=-1)
            gains.append(torch.rad2deg(target_norm - residual))
        tau_active = target_tau.abs().reshape(-1) > 1e-7
        if bool(tau_active.any()):
            tau_errors.append(1000.0 * (output_a.predictions[key].delta_tau - target_tau).abs().reshape(-1)[tau_active])
    metrics["active_gain_vs_zero_deg"] = torch.cat(gains).mean().item() if gains else float("nan")
    metrics["active_tau_error_ms"] = torch.cat(tau_errors).mean().item() if tau_errors else float("nan")
    return metrics


precheck_history = []
if RUN_STOCHASTIC_PRECHECK:
    assert STOCHASTIC_PRECHECK_STEPS > 0 and PRECHECK_PRINT_EVERY > 0
    sample_indices = np.linspace(0, len(train_dataset) - 1, min(BATCH_SIZE, 8), dtype=int)
    precheck_base = window_batch_to(collate_windows([train_dataset[int(i)] for i in sample_indices]), DEVICE, dtype=MODEL_DTYPE)
    load_training_checkpoint(RESUME_CHECKPOINT, model=model, map_location=DEVICE, restore_rng_state=False)
    saved_source_state = {name: weight.detach().cpu().clone() for name, weight in model.state_dict().items()}
    saved_model_mode = model.training
    precheck_optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_generator = make_augmentation_generator(TRAIN_PAIR_SEED + 200000)
    precheck_frame_sampler = CommonBodyFrameRandomizer(common_frame_config(180.0, probability=PRECHECK_COMMON_YAW_PROBABILITY))

    try:
        for step in range(STOCHASTIC_PRECHECK_STEPS + 1):
            if step > 0:
                model.train()
                precheck_optimizer.zero_grad(set_to_none=True)
                groups = topology_groups(precheck_base, train_generator, mode="mixture")
                full_count = precheck_base.current_calibration[CALIBRATION_KEYS[0]].transform.shape[0]
                for topology, subgroup in groups:
                    n = next(iter(subgroup.current_calibration.values())).transform.shape[0]
                    a, b, H, active, opposite = make_equivariant_windows(subgroup, generator=train_generator, return_negative=True, frame_randomizer=precheck_frame_sampler, physical_pipeline=context_augmentation_pipelines[topology])
                    pa, pb = topology_window_step(a).model_output, topology_window_step(b).model_output
                    pc = topology_window_step(opposite).model_output if USE_SIGNED_PAIR else None
                    loss, train_metrics = compute_equivariant_objective(a, pa, b, pb, H, active, opposite if USE_SIGNED_PAIR else None, pc)
                    (loss * n / full_count).backward()
                if GRADIENT_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                precheck_optimizer.step()

            if step == 0 or step == 1 or step % PRECHECK_PRINT_EVERY == 0 or step == STOCHASTIC_PRECHECK_STEPS:
                model.eval()
                with torch.inference_mode():
                    for mode_index, mode in enumerate(TOPOLOGY_MODES):
                        metrics = precheck_metrics_on_mode(precheck_base, mode, PRECHECK_EVAL_SEED + 100 * mode_index, precheck_frame_sampler)
                        precheck_history.append({"step": step, "topology": mode, **metrics})
                        print(f"Step {step:3d} | {mode:20s} | fixed loss={metrics['loss_supervised_selection']:7.3f} | active rotation gain={metrics['active_gain_vs_zero_deg']:+7.2f}deg | active tau error={metrics['active_tau_error_ms']:7.1f}ms | pair={metrics['loss_pair']:7.3f}", flush=True)
    finally:
        model.load_state_dict(saved_source_state)
        model.train(saved_model_mode)
        optimizer.zero_grad(set_to_none=True)
        del precheck_optimizer, train_generator, saved_source_state, precheck_base
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print("Precheck completed; 002.23 weights and the full-run optimizer were restored.")
else:
    print("Topology-mixture stochastic precheck skipped.")

if precheck_history:
    import pandas as pd
    precheck_df = pd.DataFrame(precheck_history)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for mode, group in precheck_df.groupby("topology"):
        group = group.sort_values("step")
        axes[0].plot(group.step, group.loss_supervised_selection, label=mode)
        axes[1].plot(group.step, group.active_gain_vs_zero_deg, label=mode)
        axes[2].plot(group.step, group.active_tau_error_ms, label=mode)
    axes[0].set(title="Held-fixed supervised selection loss", ylabel="Loss")
    axes[1].set(title="Active rotation gain over zero", ylabel="Gain [deg]")
    axes[2].set(title="Active clock correction error", ylabel="Error [ms]")
    for ax in axes:
        ax.set_xlabel("Precheck optimizer step");ax.grid(alpha=0.25);ax.legend(fontsize=8)
    plt.tight_layout();plt.show()

In [ ]:
RUN_FULL_TRAINING = True  # Change to False to inspect stochastic precheck before committing to 120 epochs.
print("Full topology-mixture training enabled:", RUN_FULL_TRAINING)
print("One optimizer batch retains BATCH_SIZE windows, partitioned across homogeneous topologies.")

## 11. Original minibatch/three-view epoch runner with per-window topology groups

In [ ]:
def run_equivariant_epoch(loader, *, training: bool, generator: torch.Generator, global_step: int, frame_randomizer: CommonBodyFrameRandomizer | None = None, topology_mode: str = "mixture"):
    """Train/evaluate heterogeneous topology samples in homogeneous subgroups.

    Each original DataLoader batch still contributes one optimizer-step unit. The
    loss in every subgroup is weighted by its size divided by the whole batch size.
    """
    model.train(training)
    metric_sums, metric_counts, num_samples = {}, {}, 0
    topology_counts = {mode: 0 for mode in TOPOLOGY_MODES}
    start = time.perf_counter()
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch_number, base in enumerate(loader, start=1):
            base = window_batch_to(base, DEVICE, dtype=MODEL_DTYPE)
            batches = topology_groups(base, generator, mode=topology_mode)
            batch_total = sum(next(iter(sub.current_calibration.values())).transform.shape[0] for _, sub in batches)
            if training and (batch_number - 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.zero_grad(set_to_none=True)
            group_index = (batch_number - 1) // GRADIENT_ACCUMULATION_STEPS
            accumulation_size = min(GRADIENT_ACCUMULATION_STEPS, len(loader) - group_index * GRADIENT_ACCUMULATION_STEPS)
            for topology, sub in batches:
                sample_count = next(iter(sub.current_calibration.values())).transform.shape[0]
                topology_counts[topology] += sample_count
                physical_pipeline = context_augmentation_pipelines[topology]
                if USE_SIGNED_PAIR:
                    original, rotated, H, active, opposite = make_equivariant_windows(sub, generator=generator, return_negative=True, frame_randomizer=frame_randomizer, physical_pipeline=physical_pipeline)
                else:
                    original, rotated, H, active = make_equivariant_windows(sub, generator=generator, frame_randomizer=frame_randomizer, physical_pipeline=physical_pipeline)
                    opposite = None
                out_original = topology_window_step(original).model_output
                out_rotated = topology_window_step(rotated).model_output
                out_opposite = topology_window_step(opposite).model_output if opposite is not None else None
                loss, metrics = compute_equivariant_objective(original, out_original, rotated, out_rotated, H, active, opposite, out_opposite)
                if training:
                    (loss * (sample_count / batch_total) / accumulation_size).backward()
                accumulate_metrics(metric_sums, metrics, sample_count, metric_counts)
                num_samples += sample_count
                del original, rotated, H, active, opposite, out_original, out_rotated, out_opposite, loss
            if training and (batch_number % GRADIENT_ACCUMULATION_STEPS == 0 or batch_number == len(loader)):
                if GRADIENT_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                optimizer.step()
                global_step += 1
            if training and PRINT_EVERY_BATCHES > 0 and batch_number % PRINT_EVERY_BATCHES == 0:
                print(f"batch {batch_number}/{len(loader)} | topology counts={topology_counts} | recent supervised={metrics['loss_supervised']:.4f}", flush=True)
    elapsed = time.perf_counter() - start
    epoch_metrics = normalize_metrics(metric_sums, num_samples, metric_counts)
    epoch_metrics["epoch_time_s"] = elapsed
    epoch_metrics["samples_per_s"] = num_samples / max(elapsed, 1e-9)
    for topology, count in topology_counts.items():
        epoch_metrics[f"topology/{topology}/fraction"] = count / num_samples
    return epoch_metrics, global_step


def accumulate_metrics(metric_sums, metrics, batch_size, metric_counts):
    for key, value in metrics.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + float(value) * batch_size
        metric_counts[key] = metric_counts.get(key, 0) + batch_size


def normalize_metrics(metric_sums, num_samples, metric_counts):
    if num_samples <= 0:
        raise ValueError("No examples in epoch.")
    # Dynamic per-head metrics appear only for modes containing that sensor key.
    return {key: value / metric_counts[key] for key, value in metric_sums.items()}

## 12. Experiment and topology-aware checkpoint metadata

In [ ]:
RUN_DIR = ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = RUN_DIR / "best.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last.pt"
METRICS_PATH = RUN_DIR / "metrics.jsonl"
EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME, "architecture": "PooledCalibrationModel", "model_config": model_config,
    "microbatch_size": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "windowing_config": windowing_config, "rig_calibration": rig_calibration, "measurement_dim": MEASUREMENT_DIM,
    "source_calibration_keys": CALIBRATION_KEYS,
    "topology_sampling_unit": "independent_physical_windows_grouped_into_homogeneous_subbatches", "topology_modes": TOPOLOGY_MODES, "topology_probabilities": TOPOLOGY_PROBABILITIES, "topology_configs": {mode: pipe.config.sensor_topology for mode, pipe in context_augmentation_pipelines.items()},
    "sequence_split": {"train": tuple(TRAIN_NAMES), "validation": tuple(VAL_NAMES), "test": tuple(TEST_NAMES)},
    "paired_prior_training": USE_SIGNED_PAIR,
    "pair_sampling_strategy": "single_truncated_student_t_one_calibration_per_sample_with_optional_opposite_prior",
    "pair_prior_config": pair_prior_config, "context_augmentation_config": context_augmentation_config,
    "equivariance_loss_weight": EQUIVARIANCE_LOSS_WEIGHT, "paired_view_probability": COMMON_BODY_FRAME_PROBABILITY,
    "common_frame_angle_curriculum_enabled": False, "common_frame_static_max_rotation_deg": COMMON_BODY_FRAME_MAX_ROTATION_DEG,
    "validation_uses_full_selected_axis_range": VALIDATE_WITH_FULL_SELECTED_AXIS_RANGE,
    "common_frame_bounded_angle_distribution": "uniform_angle_uniform_axis",
    "active_rotation_supervision_weight": ACTIVE_ROTATION_SUPERVISION_WEIGHT, "rotated_active_multiplier": ROTATED_ACTIVE_MULTIPLIER, "signed_pair_enabled": USE_SIGNED_PAIR, "signed_pair_weight": PAIR_LOSS_WEIGHT, "signed_pair_variant": "cross_frame_rotated_vs_transformed_opposite_rotation", "opposite_view_supervision_weight": OPPOSITE_VIEW_SUPERVISION_WEIGHT,
    "paired_view_transform": "shared_left_rotation_only_SO3",
    "training_objective": "supervised_views_plus_rotated_active_rotation_plus_equivariance_plus_cross_frame_signed_rotation",
    "equivariance_components": "rotation_and_translation_adjoint_plus_tau_identity",
    "pair_rotation_max_rad": PAIR_ROTATION_MAX_RAD, "pair_time_offset_max_s": PAIR_TIME_OFFSET_MAX_S,
    "initialization_checkpoint": str(RESUME_CHECKPOINT), "checkpoint_selection_metric": "loss_supervised_selection",
    "true_calibration_rotation_max_rad": TRUE_CALIBRATION_ROTATION_MAX_RAD,
    "true_calibration_time_offset_max_s": TRUE_CALIBRATION_TIME_OFFSET_MAX_S,
    "seeds": {"random": RANDOM_SEED, "train": TRAIN_PAIR_SEED, "validation": VALIDATION_PAIR_SEED, "test": TEST_PAIR_SEED},
}
LOGGER_EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME, "architecture": "PooledCalibrationModel", "calibration_keys": list(CALIBRATION_KEYS),
    "training_objective": EXPERIMENT_SNAPSHOT["training_objective"], "equivariance_loss_weight": EQUIVARIANCE_LOSS_WEIGHT,
    "paired_view_probability": COMMON_BODY_FRAME_PROBABILITY, "topology_probabilities": str(dict(zip(TOPOLOGY_MODES, TOPOLOGY_PROBABILITIES))), "validation_uses_full_selected_axis_range": VALIDATE_WITH_FULL_SELECTED_AXIS_RANGE, "prior_rotation_max_deg": float(np.rad2deg(PAIR_ROTATION_MAX_RAD)),
    "true_calibration_rotation_max_deg": float(np.rad2deg(TRUE_CALIBRATION_ROTATION_MAX_RAD)),
    "active_rotation_supervision_weight": ACTIVE_ROTATION_SUPERVISION_WEIGHT, "rotated_active_multiplier": ROTATED_ACTIVE_MULTIPLIER, "signed_pair_enabled": USE_SIGNED_PAIR, "signed_pair_weight": PAIR_LOSS_WEIGHT, "signed_pair_variant": "cross_frame_rotation",
    "source_checkpoint": str(RESUME_CHECKPOINT), "checkpoint_selection_metric": "loss_supervised_selection",
}
train_equiv_generator = make_augmentation_generator(TRAIN_PAIR_SEED)
if precheck_history:
    import pandas as pd
    pd.DataFrame(precheck_history).to_csv(RUN_DIR / "topology_stochastic_precheck.csv", index=False)
    print("Saved stochastic precheck:", RUN_DIR / "topology_stochastic_precheck.csv")


## 13. Load 002.23 calibration-only weights (fresh optimizer and scheduler)

In [ ]:
start_epoch = 0
global_step = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT is not None:
    checkpoint = load_training_checkpoint(
        RESUME_CHECKPOINT,
        model=model,
        # Initialize model weights only. The optimizer, scheduler and RNG state are fresh.
        # optimizer=optimizer,
        # scheduler=scheduler,
        map_location=DEVICE,
        restore_rng_state=False,
    )
    # start_epoch = int(checkpoint.get("epoch", 0))
    # global_step = int(checkpoint.get("global_step", 0))
    # best_validation_loss = float(checkpoint.get("extra_state", {}).get("best_validation_loss", float("inf")))

    print("Initialized model weights from:", RESUME_CHECKPOINT)
    print("start epoch:", start_epoch)
    print("global step:", global_step)

## 14. Full topology-mixture training, held-out validation, and checkpoint selection

In [ ]:
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
history = []
print("RUN_FULL_TRAINING:", RUN_FULL_TRAINING, "| source:", RESUME_CHECKPOINT)
if RUN_FULL_TRAINING:
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    with MetricsLogger(METRICS_PATH, append=False) as logger:
        logger.log_config(model_config, name="model_config")
        logger.log_config(context_augmentation_config, name="context_augmentation_config")
        logger.log_config(pair_prior_config, name="prior_config")
        logger.log_config(LOGGER_EXPERIMENT_SNAPSHOT, name="experiment")
        for epoch in range(start_epoch, NUM_EPOCHS):
            train_metrics, global_step = run_equivariant_epoch(train_loader, training=True, generator=train_equiv_generator, global_step=global_step)
            val_generator = make_augmentation_generator(VALIDATION_PAIR_SEED)
            val_sampler = full_range_frame_view_randomizer if VALIDATE_WITH_FULL_SELECTED_AXIS_RANGE else frame_view_randomizer
            val_metrics, _ = run_equivariant_epoch(val_loader, training=False, generator=val_generator, global_step=global_step, frame_randomizer=val_sampler)
            selection_loss = val_metrics["loss_supervised_selection"]
            lr_used = optimizer.param_groups[0]["lr"]
            scheduler.step()
            row = {"epoch": epoch + 1, "global_step": global_step, "learning_rate": lr_used, "train_topology_mixture": "50/25/25", **{f"train/{k}": v for k, v in train_metrics.items()}, **{f"val/{k}": v for k, v in val_metrics.items()}}
            history.append(row)
            logger.log_metrics({**train_metrics, "learning_rate": lr_used}, split="train", epoch=epoch + 1, step=global_step)
            logger.log_metrics(val_metrics, split="validation", epoch=epoch + 1, step=global_step)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} [topologies 50/25/25, fixed Z-yaw p={COMMON_BODY_FRAME_PROBABILITY:g}]: train supervised={train_metrics['loss_supervised']:.4f} active={train_metrics['loss_active_supervised']:.4f} equiv={train_metrics['loss_equivariance']:.4f} pair={train_metrics['loss_pair']:.4f} | val supervised={val_metrics['loss_supervised']:.4f} (orig={val_metrics['loss_original']:.4f}, rotated={val_metrics['loss_rotated']:.4f}) active={val_metrics['loss_active_supervised']:.4f} pair={val_metrics['loss_pair']:.4f} | val activeRot orig={val_metrics['original/active_rotation_error_deg']:.2f}deg rotated={val_metrics['rotated/active_rotation_error_deg']:.2f}deg | gapPred={val_metrics.get('pair/predicted_rotation_gap_deg', 0.0):.1f}deg/gapTrue={val_metrics.get('pair/target_rotation_gap_deg', 0.0):.1f}deg | activeMagPredRot={val_metrics.get('rotated/active_rotation_prediction_deg', 0.0):.1f}deg/target={val_metrics.get('rotated/active_rotation_target_deg', 0.0):.1f}deg | select={selection_loss:.4f} lr={lr_used:.2e}", flush=True)
            if selection_loss < best_validation_loss:
                best_validation_loss = selection_loss
                save_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_shuffle": train_shuffle_generator, "train_pair": train_equiv_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
                print("Saved best checkpoint.")
            if (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0 or epoch + 1 == NUM_EPOCHS:
                save_training_checkpoint(LAST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_shuffle": train_shuffle_generator, "train_pair": train_equiv_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Training disabled in configuration. The source checkpoint and smoke checks are unaffected.")

## 15. Checkpoint snapshot verification

In [ ]:
from obscalib.training.checkpoint import read_training_checkpoint
for checkpoint_path in (BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH):
    if not checkpoint_path.is_file():
        print("Not yet created:", checkpoint_path)
        continue
    saved = read_training_checkpoint(checkpoint_path, map_location="cpu")
    snapshot = (saved.get("extra_state") or {}).get("experiment", {})
    assert snapshot.get("run_name") == RUN_NAME
    assert snapshot.get("training_objective") == EXPERIMENT_SNAPSHOT["training_objective"]
    assert isinstance(snapshot.get("model_config"), PooledCalibrationModelConfig)
    assert isinstance(snapshot.get("windowing_config"), WindowingConfig)
    print("Verified:", checkpoint_path.name, "epoch:", saved.get("epoch"))

## 16. Learning curves and empirical topology frequencies

In [ ]:
if history:
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    for phase in ("train", "val"):
        for key in ("loss_original", "loss_rotated", "loss_supervised_selection"):
            axes[0].plot(epochs, [r[f"{phase}/{key}"] for r in history], label=f"{phase} {key}")
        for key in ("loss_equivariance", "loss_pair", "loss_active_supervised"):
            axes[1].plot(epochs, [r[f"{phase}/{key}"] for r in history], label=f"{phase} {key}")
    for view in ("original", "rotated"):
        axes[2].plot(epochs, [r[f"val/{view}/active_rotation_error_deg"] for r in history], label=f"validation {view} active rotation")
    axes[0].set_ylabel("Supervised loss")
    axes[1].set_ylabel("Auxiliary raw loss")
    axes[2].set_ylabel("Active correction error [deg]")
    # Topology probabilities stay fixed throughout training; there are no yaw stages.
    axes[2].set_xlabel("Epoch")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.figure(figsize=(10, 3))
    plt.plot(epochs, [r["learning_rate"] for r in history])
    plt.title("Learning rate, fresh optimizer from 002.23")
    plt.grid(alpha=0.25)
    plt.show()
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for mode in TOPOLOGY_MODES:
        ax.plot(epochs, [r.get(f"train/topology/{mode}/fraction", float("nan")) for r in history], label=f"train {mode}")
        ax.axhline(TOPOLOGY_PROBABILITIES[TOPOLOGY_MODES.index(mode)], linewidth=0.8, linestyle="--")
    ax.set(xlabel="Epoch", ylabel="Fraction of physical windows", title="Per-window topology sampling distribution")
    ax.grid(alpha=0.25); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 17. Fair held-out comparison of checkpoint 002.23 against the newly trained model

For each topology, we reset the augmentation generator before each checkpoint. Each model therefore sees the **same KAIST held-out windows, sensor mountings, true calibrations, one-sensor prior faults and static yaw views**. Report all three topologies separately and preserve performance on unmodified KAIST as a regression-control condition.

In [ ]:
import pandas as pd
COMPARE_TO_SOURCE_CHECKPOINT = True


def evaluate_topology_distribution(checkpoint_path, label):
    load_training_checkpoint(checkpoint_path, model=model, map_location=DEVICE, restore_rng_state=False)
    model.eval()
    rows = []
    with torch.inference_mode():
        for mode_index, mode in enumerate(TOPOLOGY_EVAL_MODES):
            generator = make_augmentation_generator(TEST_PAIR_SEED + 901 + 10000 * mode_index)
            metrics, _ = run_equivariant_epoch(test_loader, training=False, generator=generator, global_step=0, frame_randomizer=full_range_frame_view_randomizer, topology_mode=mode)
            rows.append({"checkpoint": label, "topology": mode, "active_rotation_error_deg": metrics["original/active_rotation_error_deg"], "rotated_active_rotation_error_deg": metrics["rotated/active_rotation_error_deg"], "active_tau_error_ms": metrics["original/active_tau_error_ms"], "selection_loss": metrics["loss_supervised_selection"], "supervised_loss": metrics["loss_supervised"], "pair_loss": metrics["loss_pair"], "equivariance_loss": metrics["loss_equivariance"], "original_rotation_error_imu_deg": metrics.get("original/imu/rotation_error_deg", float("nan")), "original_rotation_error_lidar_deg": metrics.get("original/lidar/rotation_error_deg", float("nan")), "original_rotation_error_lidar_right_deg": metrics.get("original/lidar_right/rotation_error_deg", float("nan")), "original_rotation_error_imu_synth_deg": metrics.get("original/imu_synth/rotation_error_deg", float("nan"))})
    return pd.DataFrame(rows)


if BEST_CHECKPOINT_PATH.is_file():
    result_source = evaluate_topology_distribution(RESUME_CHECKPOINT, "002.23 source") if COMPARE_TO_SOURCE_CHECKPOINT else pd.DataFrame()
    result_new = evaluate_topology_distribution(BEST_CHECKPOINT_PATH, "new best")
    topology_eval_df = pd.concat([result_source, result_new], ignore_index=True)
    display(topology_eval_df.round(3))
    if not result_source.empty:
        paired = result_new.set_index("topology").select_dtypes(include="number") - result_source.set_index("topology").select_dtypes(include="number")
        print("New minus 002.23. Negative is better for error/loss metrics:")
        display(paired.round(3))
        paired.reset_index().to_csv(RUN_DIR / "topology_mixture_heldout_delta_vs_00223.csv", index=False)
    topology_eval_df.to_csv(RUN_DIR / "topology_mixture_heldout_checkpoint_comparison.csv", index=False)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, metric, title in zip(axes, ("active_rotation_error_deg", "active_tau_error_ms", "selection_loss"), ("Active rotation error", "Active clock error", "Supervised selection loss")):
        values = topology_eval_df.pivot(index="topology", columns="checkpoint", values=metric).reindex(TOPOLOGY_MODES)
        values.plot.bar(ax=ax, rot=15)
        ax.set(title=title, ylabel="deg" if metric.endswith("_deg") else "ms" if metric.endswith("_ms") else "Loss")
        ax.grid(alpha=0.2, axis="y")
    fig.tight_layout();plt.show()
else:
    print("Best checkpoint not found. Finish training before the held-out checkpoint comparison.")

## 18. Controlled KAIST sensor-topology ablation, identical faults and mountings for both checkpoints

This recreates the causal test from notebook 37 **without random training augmentations**. In all modes, raw KAIST motion stays identical. The original IMU and the retained real LiDAR receive exactly the same ±15° or ±45° body-frame faults. The synthetic IMU also receives a separate fault test. Each physical window and synthetic mounting is **cached once and reused** for both models. Use geodesic SO(3) correction gain, signed-response slope, and fixed-point bias as the primary metrics.

In [ ]:
from scipy.spatial.transform import Rotation

CONTROL_WINDOWS_PER_SEQUENCE = 4
CONTROL_FAULT_ANGLES_DEG = (0.0, 15.0, 45.0)
CONTROL_FAULT_AXES = (0, 2)
CONTROL_INCLUDE_SYNTHETIC_HEAD = True


def control_make_candidate(states, sensor, axis, signed_angle_deg):
    result = {key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for key, state in states.items()}
    perturb = torch.zeros((result[sensor].transform.shape[0], 3), device=result[sensor].transform.device, dtype=result[sensor].transform.dtype)
    perturb[:, axis] = np.deg2rad(signed_angle_deg)
    T = result[sensor].transform.clone()
    T[:, :3, :3] = so3_exp(perturb) @ T[:, :3, :3]
    result[sensor] = CalibrationState(transform=T, time_offset=result[sensor].time_offset)
    return result


def control_rotation_gain(truth, candidate, sensor, predicted_phi):
    R_true = truth[sensor].transform[0, :3, :3].detach().cpu().numpy().astype(float)
    R_initial = candidate[sensor].transform[0, :3, :3].detach().cpu().numpy().astype(float)
    R_corrected = Rotation.from_rotvec(predicted_phi).as_matrix() @ R_initial
    before = float(np.rad2deg(Rotation.from_matrix(R_true @ R_initial.T).magnitude()))
    after = float(np.rad2deg(Rotation.from_matrix(R_true @ R_corrected.T).magnitude()))
    return before - after, after


# Cache the data and synthesized mountings so they cannot differ by checkpoint.
control_physical_windows = []
for sequence in TEST_NAMES:
    sequence_indices = np.asarray(test_dataset.window_indices_for_sequence(sequence), dtype=int)
    if not len(sequence_indices):
        continue
    positions = np.unique(np.linspace(0, len(sequence_indices) - 1, min(CONTROL_WINDOWS_PER_SEQUENCE, len(sequence_indices))).round().astype(int))
    for position in positions:
        index = int(sequence_indices[position])
        base = window_batch_to(collate_windows([test_dataset[index]]), DEVICE, dtype=MODEL_DTYPE)
        for mode_index, mode in enumerate(TOPOLOGY_MODES):
            physical, _ = context_augmentation_pipelines[mode].sensor_topology_augmenter(base, generator=make_augmentation_generator(TEST_PAIR_SEED + index * 10 + mode_index))
            assert topology_mode_of(physical) == mode
            control_physical_windows.append((sequence, index, mode, physical))


def control_run_checkpoint(checkpoint_path, checkpoint_label):
    load_training_checkpoint(checkpoint_path, model=model, map_location=DEVICE, restore_rng_state=False)
    model.eval()
    rows = []
    with torch.inference_mode():
        for sequence, window_index, mode, physical in tqdm(control_physical_windows, desc=f"Controlled topologies: {checkpoint_label}"):
            retained_sensors = tuple(physical.current_calibration)
            for sensor in retained_sensors:
                if sensor == "imu_synth" and not CONTROL_INCLUDE_SYNTHETIC_HEAD:
                    continue
                for angle in CONTROL_FAULT_ANGLES_DEG:
                    for axis in CONTROL_FAULT_AXES:
                        for sign in ((+1,) if angle == 0 else (-1, +1)):
                            candidate = control_make_candidate(physical.current_calibration, sensor, axis, sign * angle)
                            candidate_window = clone_window_with_calibration(physical, candidate)
                            output = topology_window_step(candidate_window).model_output
                            delta_phi = output.predictions[sensor].delta_xi[0, :3].detach().cpu().numpy().astype(float)
                            gain, after = control_rotation_gain(physical.current_calibration, candidate, sensor, delta_phi)
                            rows.append({"checkpoint": checkpoint_label, "sequence": sequence, "window": window_index, "topology": mode, "sensor": sensor, "role": "imu" if sensor.startswith("imu") else "lidar", "axis": "xyz"[axis], "sign": sign, "fault_angle_deg": angle, "fault_signed_deg": sign * angle, "predicted_axis_rad": float(delta_phi[axis]), "predicted_norm_deg": float(np.rad2deg(np.linalg.norm(delta_phi))), "gain_deg": gain, "residual_error_deg": after})
    return pd.DataFrame(rows)


if BEST_CHECKPOINT_PATH.is_file():
    source_control = control_run_checkpoint(RESUME_CHECKPOINT, "002.23 source")
    new_control = control_run_checkpoint(BEST_CHECKPOINT_PATH, "new best")
    control_results = pd.concat((source_control, new_control), ignore_index=True)
    shared_identifiers = ["sequence", "window", "topology", "sensor", "axis", "sign", "fault_angle_deg"]
    assert not control_results.duplicated(shared_identifiers + ["checkpoint"]).any()
    control_results.to_csv(RUN_DIR / "topology_fixed_faults_both_checkpoints.csv", index=False)
    display(control_results.groupby(["checkpoint", "topology", "role", "sensor", "fault_angle_deg"]).agg(n=("gain_deg", "size"), mean_gain_deg=("gain_deg", "mean"), median_gain_deg=("gain_deg", "median"), mean_residual_deg=("residual_error_deg", "mean"), mean_prediction_magnitude_deg=("predicted_norm_deg", "mean")).round(3))
    new_gain = new_control.set_index(shared_identifiers)["gain_deg"]
    source_gain = source_control.set_index(shared_identifiers)["gain_deg"]
    paired_gain = (new_gain - source_gain).rename("gain_improvement_over_00223_deg").reset_index()
    print("Paired gain improvement on EXACTLY the same motions/mountings/faults (positive is better):")
    display(paired_gain.groupby(["topology", "role", "sensor", "fault_angle_deg"])["gain_improvement_over_00223_deg"].agg(["mean", "median", "count"]).round(3))
    paired_gain.to_csv(RUN_DIR / "topology_fixed_faults_paired_gain_change.csv", index=False)
else:
    print("Controlled ablation skipped; best checkpoint unavailable.")

## 18b. Topology-specific signed correction sensitivity and checkpoint improvement plots

In [ ]:
if "control_results" not in globals():
    print("Run section 18 after training to show the controlled-topology plots.")
else:
    import pandas as pd
    from matplotlib.ticker import MaxNLocator

    # Signed response slope close to +1 means the model reverses its correction
    # when we reverse a known injected fault. Use fault angle 15/45 only.
    nonzero = control_results[control_results.fault_angle_deg > 0]
    slope_index = ["checkpoint", "sequence", "window", "topology", "sensor", "role", "axis", "fault_angle_deg"]
    slope = nonzero.pivot(index=slope_index, columns="sign", values="predicted_axis_rad")
    radians = np.deg2rad(slope.index.get_level_values("fault_angle_deg").to_numpy(dtype=float))
    slope["signed_slope"] = (slope[-1] - slope[1]) / (2 * radians)
    slope_df = slope.reset_index()
    slope_df.to_csv(RUN_DIR / "topology_signed_response_slopes.csv", index=False)
    display(slope_df.groupby(["checkpoint", "topology", "role", "sensor", "fault_angle_deg"]).signed_slope.agg(["mean", "median", "std"]).round(3))

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    for column, mode in enumerate(TOPOLOGY_MODES):
        for checkpoint, group in control_results[(control_results.topology == mode) & (control_results.fault_angle_deg > 0)].groupby("checkpoint"):
            for role, role_group in group.groupby("role"):
                summary = role_group.groupby("fault_angle_deg").gain_deg.mean().sort_index()
                axes[0, column].plot(summary.index, summary.values, marker="o", label=f"{checkpoint}/{role}")
        axes[0, column].plot([0, 45], [0, 45], linestyle="--", color="gray", linewidth=1, label="Ideal gain")
        axes[0, column].set(title=mode, xlabel="Fault angle [deg]", ylabel="Mean geodesic correction gain [deg]")
        for (checkpoint, role), group in slope_df[slope_df.topology == mode].groupby(["checkpoint", "role"]):
            summary = group.groupby("fault_angle_deg").signed_slope.mean().sort_index()
            axes[1, column].plot(summary.index, summary.values, marker="o", label=f"{checkpoint}/{role}")
        axes[1, column].axhline(1.0, color="gray", linestyle="--", linewidth=1)
        axes[1, column].set(xlabel="Fault angle [deg]", ylabel="Mean signed response slope")
        for row in range(2):
            axes[row, column].grid(alpha=0.25)
            axes[row, column].legend(fontsize=7)
            axes[row, column].xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.suptitle("002.23 versus topology-trained checkpoint: controlled single-sensor faults")
    fig.tight_layout(); fig.savefig(RUN_DIR / "topology_controlled_gain_and_slopes.png", dpi=160); plt.show()

    # Fixed point: a zero-error candidate should not trigger a correction.
    zero_cases = control_results[control_results.fault_angle_deg == 0]
    print("Nominal-calibration update magnitude. Smaller is better:")
    display(zero_cases.groupby(["checkpoint", "topology", "sensor"]).predicted_norm_deg.agg(["mean", "median", "max"]).round(3))

## 19. Original-KAIST exact-truth fixed-point regression check

Retain notebook 39's original KAIST fixed-point test. In addition, the controlled ablation's zero-fault results in section 18b check all three topologies and compare both checkpoints.

In [ ]:
if BEST_CHECKPOINT_PATH.is_file():
    load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)

sequence_indices = [i for i in range(len(test_dataset)) if test_dataset.sequence_name_for_window(i) == TEST_NAMES[0]]
FIXED_POINT_NUM_WINDOWS = min(100, len(sequence_indices))
fixed_point_positions = np.unique(np.linspace(0, len(sequence_indices) - 1, FIXED_POINT_NUM_WINDOWS).round().astype(int))
fixed_point_indices = np.asarray(sequence_indices)[fixed_point_positions]

fixed_point_results = {calibration_key: {"delta_xi": [], "delta_tau": []} for calibration_key in CALIBRATION_KEYS}

model.eval()

with torch.no_grad():
    for dataset_index in fixed_point_indices:
        window = window_batch_to(collate_windows([test_dataset[int(dataset_index)]]), DEVICE, dtype=MODEL_DTYPE)
        exact_true_calibration = {calibration_key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for calibration_key, state in window.current_calibration.items()}
        result = topology_window_step(window, calibration=exact_true_calibration)

        for calibration_key, prediction in result.model_output.predictions.items():
            fixed_point_results[calibration_key]["delta_xi"].append(prediction.delta_xi[0].detach().cpu())
            fixed_point_results[calibration_key]["delta_tau"].append(prediction.delta_tau[0, 0].detach().cpu())

for calibration_key in CALIBRATION_KEYS:
    fixed_point_results[calibration_key]["delta_xi"] = torch.stack(fixed_point_results[calibration_key]["delta_xi"]).numpy()
    fixed_point_results[calibration_key]["delta_tau"] = torch.stack(fixed_point_results[calibration_key]["delta_tau"]).numpy()

component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")

print("=" * 120)
print("EXACT-TRUTH FIXED-POINT DIAGNOSTIC")
print("=" * 120)

for calibration_key in CALIBRATION_KEYS:
    result = fixed_point_results[calibration_key]
    prediction = np.concatenate([result["delta_xi"], result["delta_tau"][:, None]], axis=1)

    print(calibration_key)
    print(f"{'component':8s} {'mean':>16s} {'mean |x|':>16s} {'RMSE':>16s} {'std':>16s} {'range':>16s}")

    for component_index, component_name in enumerate(component_names):
        values = prediction[:, component_index]

        if component_index < 3:
            values, unit = np.rad2deg(values), "deg"
        elif component_index < 6:
            values, unit = 100.0 * values, "cm"
        else:
            values, unit = 1000.0 * values, "ms"

        mean = np.mean(values)
        mean_abs = np.mean(np.abs(values))
        rmse = np.sqrt(np.mean(values ** 2))
        std = np.std(values)
        value_range = np.max(values) - np.min(values)

        print(f"{component_name:8s} {mean:+12.6f} {unit:3s} {mean_abs:12.6f} {unit:3s} {rmse:12.6f} {unit:3s} {std:12.6f} {unit:3s} {value_range:12.6f} {unit:3s}")

    print()

## 20. Interpretation and next evaluation

The first checkpoint comparison is **paired within each topology**: reset the RNG and use the identical held-out KAIST windows and synthetic sensor mountings for 002.23 and the newly trained model. Inspect rotation gain on actively perturbed sensors, nominal fixed points, temporal-offset errors, and any regression on original 1-IMU/2-LiDAR KAIST.

For a more controlled topology causal comparison, section 18 also injects fixed ±15° and ±45° errors into shared physical sensor heads with the same held-out KAIST windows, not random training-distribution perturbations.

Caveat: simulated IMU2 is derived from KAIST IMU1, not independently recorded. High synthetic-topology performance does not guarantee NewCollege transfer, but directly tests the 1I2L → 2I1L failure discovered in notebook 37.

In [ ]:
print("Run:", RUN_NAME)
print("Starting weights:", RESUME_CHECKPOINT)
print("Fixed true topology proportions:", dict(zip(TOPOLOGY_MODES, TOPOLOGY_PROBABILITIES)))
print("Static common-body Z-yaw probability:", COMMON_BODY_FRAME_PROBABILITY)
print("Per sample, only one calibration prior is faulty:", pair_prior_config.one_calibration_per_sample)
print("Best checkpoint:", BEST_CHECKPOINT_PATH, "exists:", BEST_CHECKPOINT_PATH.is_file())
if history:
    row = min(history, key=lambda result: result["val/loss_supervised_selection"])
    print("Best epoch:", row["epoch"], "selection loss:", row["val/loss_supervised_selection"])

In [ ]:
if DEVICE.type == "cuda": torch.cuda.empty_cache()